In [ ]:
import xarray as xr

# from dask.distributed import Client, LocalCluster
from genpp.configs import register_resolvers
from genpp.data import (
    FORECAST_ENS_FLAT_AGG_PATH,
    OBSERVATIONS_FLAT_PATH,
)
from genpp.data.utils import flatten_levels

register_resolvers()

In [ ]:
ens = xr.open_zarr(FORECAST_ENS_FLAT_AGG_PATH, consolidated=True)
obs = xr.open_zarr(OBSERVATIONS_FLAT_PATH, consolidated=True)

In [ ]:
import hydra
from hydra import compose, initialize_config_dir

from genpp import BASE_DIR

CONFIG_DIR = BASE_DIR / "configs"
with initialize_config_dir(version_base=None, config_dir=str(CONFIG_DIR)):
    cfg = compose(config_name="base_drn")

In [134]:
cfg.data.module.dataset_config

{'train': {'x_kwargs': {'input_dims': {'feature': '${data.features.input_concat}', 'longitude': '${data.spatial.original.longitude}', 'latitude': '${data.spatial.original.latitude}'}, 'batch_dims': {'time': 1, 'prediction_timedelta': 1}}, 'x_transform': '${data.spatial.x_transform}', 'y_transform': '${data.spatial.y_transform}', 'slice': '${data.splits.train}'}, 'val': {'x_kwargs': {'input_dims': {'feature': '${data.features.input_concat}', 'longitude': '${data.spatial.original.longitude}', 'latitude': '${data.spatial.original.latitude}'}, 'batch_dims': {'time': 1, 'prediction_timedelta': 1}}, 'x_transform': '${data.spatial.x_transform}', 'y_transform': '${data.spatial.y_transform}', 'slice': '${data.splits.val}'}, 'test': {'x_kwargs': {'input_dims': {'feature': '${data.features.input_concat}', 'longitude': '${data.spatial.original.longitude}', 'latitude': '${data.spatial.original.latitude}'}, 'batch_dims': {'time': 1, 'prediction_timedelta': 1}}, 'x_transform': '${data.spatial.x_trans

In [ ]:
x_da = xr.open_zarr(FORECAST_ENS_FLAT_AGG_PATH, consolidated=True)

In [ ]:
x_select_variables = list(cfg.data.x_select_variables)
# Select only specified variables
if x_select_variables:
    x_da = x_da[x_select_variables]

In [ ]:
# Turn into DataArray
x_da = flatten_levels(x_da, level_dim="statistic", interleave=False)
x_da = x_da.to_dataarray(dim="feature")

In [ ]:
x_preprocessing = hydra.utils.instantiate(cfg.data.module.x_preprocessing)

In [ ]:
train_slice = hydra.utils.instantiate(cfg.data.module.dataset_config.train.slice)

In [ ]:
# Apply preprocessing to x data
if x_preprocessing:
    for preprocessor in x_preprocessing:
        # Preprocessors work on dataarrays
        preprocessor.fit(x_da.sel(time=train_slice))
        if not preprocessor.fit_only:
            x_da = preprocessor.preprocess(x_da)

In [ ]:
# Process Y data
y_da = xr.open_zarr(OBSERVATIONS_FLAT_PATH, consolidated=True)
y_select_variables = list(cfg.data.y_select_variables)
if y_select_variables:
    y_da = y_da[y_select_variables]

# Turn into DataArray (there are no levels to flatten here)
y_da = y_da.to_dataarray(dim="feature")

# Apply preprocessing to y data
y_preprocessing = hydra.utils.instantiate(cfg.data.module.y_preprocessing)
if y_preprocessing:
    for preprocessor in y_preprocessing:
        preprocessor.fit(y_da.sel(time=train_slice))
        if not preprocessor.fit_only:
            y_da = preprocessor.preprocess(y_da)

In [ ]:
dataset_config = hydra.utils.instantiate(cfg.data.module.dataset_config)
dataset_config.train.x_kwargs

In [ ]:
import tempfile
from pathlib import Path
from typing import Any, Dict, List, Tuple

import torch
import xarray as xr
import xbatcher
from tqdm import tqdm
from xbatcher.loaders.torch import to_tensor


def _cache_data(
    x_da: xr.DataArray,
    y_da: xr.DataArray,
    time_slices: Dict[str, slice],
    cache_dir: Path,
    batch_kwargs: dict,
) -> Tuple[Any, Dict[str, List[int]]]:
    """
    Pre-process xarray data and save as PyTorch tensor files.

    Args:
        x_da: Input xarray DataArray
              (feature: 63, time: 3651, prediction_timedelta: 5, latitude: 31, longitude: 37)
        y_da: Target xarray DataArray
              (feature: 2, time: 7304, latitude: 31, longitude: 37)
        time_slices: Dictionary mapping split names to time slices
        cache_dir: Directory to save cached files
        batch_kwargs: Batching configuration from xbatcher

    Returns:
        Tuple of (path_to_saved_dict, split_indices)
        The saved dict contains keys: "x", "y", "prediction_timedelta"
    """
    assert x_da.feature.shape[0] == batch_kwargs["input_dims"]["feature"]

    all_x, all_y, all_dt = [], [], []
    split_indices: Dict[str, List[int]] = {}
    current_idx = 0

    for split_name, time_slice in time_slices.items():
        # We split both datasets right here to prevent leaking data
        x_split = x_da.sel(time=time_slice)
        y_split = y_da.sel(time=time_slice)

        # These batch kwargs are fixed so that the batches are random
        batch_kwargs["batch_dims"]["time"] = 1
        batch_kwargs["batch_dims"]["prediction_timedelta"] = 1

        gen = xbatcher.BatchGenerator(x_split, **batch_kwargs)

        split_batch_indices = []
        for x_batch in tqdm(gen):
            x_tensor = to_tensor(x_batch)
            t0 = x_batch.time.values[0]
            dt = x_batch.prediction_timedelta.values[0]
            y_t = t0 + dt
            if y_t not in y_split.time.values:
                # This data is not includeded as it would be in another split
                # thus leaking data
                continue
            y_batch = y_split.sel(time=y_t, longitude=x_batch.longitude, latitude=x_batch.latitude)
            y_tensor = to_tensor(y_batch.compute())

            if y_tensor.dim() == 3:
                y_tensor = y_tensor.unsqueeze(0).unsqueeze(0)

            tensor_dt = torch.tensor([dt], dtype=torch.float32)

            all_x.append(x_tensor)
            all_y.append(y_tensor)
            all_dt.append(tensor_dt)

            split_batch_indices.append(current_idx)
            current_idx += 1

        split_indices[split_name] = split_batch_indices

    if not all_x:
        raise ValueError("No batches generated from the data")

    x_tensor = torch.cat(all_x, dim=0)
    y_tensor = torch.cat(all_y, dim=0)
    dt_tensor = torch.cat(all_dt, dim=0)

    temp_file = tempfile.NamedTemporaryFile(dir=cache_dir, suffix="_tensor.pt", delete=False)
    torch.save({"x": x_tensor, "y": y_tensor, "prediction_timedelta": dt_tensor}, temp_file.name)
    return temp_file.name, split_indices

In [ ]:
# Define time slices for splits
time_slices = {
    "train": slice("2018-01-03", "2018-02-03"),
    "val": slice("2021-01-01", "2021-01-10"),
    "test": slice("2022-01-01", "2022-01-10"),
}

cache_dir = FORECAST_ENS_FLAT_AGG_PATH.with_name("tmp")

# Preprocess and save x data
tmp, split_indices = _cache_data(
    x_da, y_da, time_slices, cache_dir, batch_kwargs=dataset_config.train.x_kwargs
)

# Store metadata in a temporary file
cache_metadata = {
    "tmp_path": tmp,
    "split_indices": split_indices,
}

In [69]:
# Store reverse modules for later use
if x_preprocessing:
    x_reverseModules = [
        rm
        for preprocessor in x_preprocessing
        if (rm := preprocessor.get_reverse_module()) is not None
    ]
if y_preprocessing:
    y_reverseModules = [
        rm
        for preprocessor in y_preprocessing
        if (rm := preprocessor.get_reverse_module()) is not None
    ]

In [70]:
import pickle

# Save metadata to temporary file
metadata_tmp = tempfile.NamedTemporaryFile(dir=cache_dir, suffix="_metadata.pkl", delete=False)

with open(metadata_tmp.name, "wb") as f:
    pickle.dump(cache_metadata, f)

already_prepared = True

In [95]:
from torch.utils.data import DataLoader, Dataset


class TransformTensorDataset(Dataset):
    """Custom dataset that applies transforms to tensors in __getitem__.

    This dataset wraps tensors and applies transforms on-the-fly during data loading,
    rather than pre-applying them during preprocessing.
    """

    def __init__(
        self,
        x_tensor: torch.Tensor,
        y_tensor: torch.Tensor,
        dt_tensor: torch.Tensor,
        x_transform: Any = None,
        y_transform: Any = None,
    ):
        """Initialize the dataset with tensors and optional transforms.

        Args:
            x_tensor: Input feature tensor
            y_tensor: Target tensor
            x_transform: Optional transform to apply to x data
            y_transform: Optional transform to apply to y data
        """
        self.x_tensor = x_tensor
        self.y_tensor = y_tensor
        self.dt_tensor = dt_tensor
        self.x_transform = x_transform
        self.y_transform = y_transform

        if len(x_tensor) != len(y_tensor):
            raise ValueError(f"Tensor lengths don't match: {len(x_tensor)} vs {len(y_tensor)}")

    def __len__(self) -> int:
        return len(self.x_tensor)

    def __getitem__(self, index) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        x = self.x_tensor[index]
        y = self.y_tensor[index]
        dt = self.dt_tensor[index]

        # Apply transforms if provided
        if self.x_transform is not None:
            x = self.x_transform(x)
        if self.y_transform is not None:
            y = self.y_transform(y)

        return x, y, dt

In [107]:
## Setup Stage
# Load cache metadata from file
with open(metadata_tmp.name, "rb") as f:
    cache_metadata = pickle.load(f)

# Load cached tensors
cached_tensors = torch.load(cache_metadata["tmp_path"])
x = cached_tensors["x"]
y = cached_tensors["y"]
dt = cached_tensors["prediction_timedelta"]


# Get transforms from metadata
x_transform = dataset_config.train.x_transform
y_transform = dataset_config.train.y_transform

# Create training dataset
train_indices = cache_metadata["split_indices"]["train"]
train_dataset = TransformTensorDataset(
    x[train_indices],
    y[train_indices],
    dt[train_indices],
    x_transform=x_transform,
    y_transform=y_transform,
)

In [ ]:
dataloader_conf = hydra.utils.instantiate(cfg.data.module.dataloader_config.train)
dataloader_conf["num_workers"] = 0  # For debugging
dataloader_conf.pop("prefetch_factor")
dataloader_conf.pop("persistent_workers")
dataloader_conf.pop("multiprocessing_context")
dl = DataLoader(train_dataset, **dataloader_conf)

In [ ]:
for batch in tqdm(dl):
    xb, yb, dtb = batch
    # print(xb.shape, yb.shape, dtb.shape)

100%|██████████| 5/5 [00:00<00:00, 106.97it/s]
